# 1. Tracing quickstart tutorial

NeMo Guardrails supports the Open Telemetry ([OTEL](https://opentelemetry.io/)) standard, to give users fine-grained visibility into server-side latency. Guardrails captures the latency of each LLM and API call, and exports this telemetry using OTEL. Latency can then be visualized by any OTEL-compatible backend, for example Grafana, Jaeger, Prometheus, SigNoz, New Relic, Datadog, Honeycomb, and many others.

This notebook walks through configuring NeMo Guardrails to export metrics in JSONL format (covered by [Documentation](https://docs.nvidia.com/nemo/guardrails/latest/user-guides/tracing/quick-start.html) here). We'll use hosted Application and Nemoguard LLMs hosted on https://build.nvidia.com/ to reduce the pre-requisite steps, for which you'll need to create an account and set the `NVIDIA_API_KEY` environment variable.

We'll run Guardrail requests in both sequential and parallel modes, showing how the parallel mode reduces end-to-end latency when more than one input or output rails are in use.

-----

## Setup

Before running any tracing with Guardrails, let's install some dependencies and import useful modules.

In [1]:
!pip install --upgrade pip

In [2]:
!pip install pandas plotly langchain_nvidia_ai_endpoints aiofiles -q

In [3]:
# Import some useful modules
import os
import pandas as pd
import plotly.express as px
import json

from typing import Dict, List, Any, Union

In [4]:
# Check the NVIDIA_API_KEY environment variable is set
assert os.getenv("NVIDIA_API_KEY"), f"Please create a key at build.nvidia.com and set the NVIDIA_API_KEY environment variable"

In [5]:
SEQUENTIAL_TRACE_FILE = "sequential_trace.jsonl"
PARALLEL_TRACE_FILE = "parallel_trace.jsonl"

In [7]:
def delete_file_if_it_exists(filename: str) -> None:
    """Check if a file exists, and delete it if so"""

    if os.path.exists(filename):
        print(f"Deleting {filename}")
        os.remove(filename)

delete_file_if_it_exists(SEQUENTIAL_TRACE_FILE)
delete_file_if_it_exists(PARALLEL_TRACE_FILE)

------

## Configuration

In this section, we'll build Guardrails Configuration objects for use in tracing. We'll use two configurations for tracing, sequential and parallel. The sequential configuration calls each input rail in-sequence. If all input rails pass, the client request is sent to the Application LLM to generate a response. Once the response is available, the output rails run one-by-one and check both user input and LLM response. If all these checks pass, the response is returned to the client.

The parallel configuration runs all input and output rails in parallel, rather than one-by-one. In this case we only have one output rail so the output parallel mode is disabled. But the three input rails can run in parallel and reduce the end-to-end latency.

### Models

We'll store the Models needed for tracing in the dictionary below. Each model entry has a `type`, `engine`, and `model` field. These fields are explained in more detail below:

* `type`: The model type identifies when and how each model is used. A reserved keyword of `main` identifies the Application LLM responsible for generating a response from the client's request. Model names other than `main` are referenced to Guardrail flows to build workflows.
* `engine`: This controls the library used to communicate with the model. The `nim` engine uses [langchain_nvidia_ai_endpoints](https://pypi.org/project/langchain-nvidia-ai-endpoints/) to communicate with Nvidia-hosted LLMs. The `openai` engine can be used to access [OpenAI-hosted models](https://platform.openai.com/docs/models).
* `model`: The name of the model used to generate a response.

In [8]:
CONFIG_MODELS: Dict[str, str] = [
        {
            'type': 'main',
            'engine': 'nim',
            'model': 'meta/llama-3.3-70b-instruct'
        },
        {
            'type': 'content_safety',
            'engine': 'nim',
            'model': 'nvidia/llama-3.1-nemoguard-8b-content-safety'
        },
        {
            'type': 'topic_control',
            'engine': 'nim',
            'model': 'nvidia/llama-3.1-nemoguard-8b-topic-control'
        }
    ]

### Rails

The Guardrails Rails section defines a workflow that executes on every client request. The high-level sections are `input` for input rails, `output` for output rails, and `config` for any additional model condfiguration. Guardrails flows reference models defined in the `CONFIG_MODELS` variable above using the `$model=<model.type>` syntax. Each rail type is explained further below.

* `input`: Input rails run on the client request only. The config below uses three classifiers to predict whether a user request is safe, on-topic, or a jailbreak attempt. These rails can be run in parallel to reduce the latency. If any of the rails predicts an unsafe input, a refusal text is returned to the user, and no LLM generation is triggered.
* `output`: Output rails run on both client request and the LLM response to that request. The example below checks whether the LLM response to the user request is safe to return. Output rails are needed as well as input because a safe request may give an unsafe response from the LLM if it interprets the request incorrectly. A refusal text is returned to the client if the response is unsafe.
* `config`: Any configuration used outside of a Langchain LLM interface is included in this section. The [Jailbreak detection model](https://build.nvidia.com/nvidia/nemoguard-jailbreak-detect) uses an embedding model as a feature-generation step, followed by a Random Forest classifier to detect a jailbreak attempt.

In [9]:
def config_rails(parallel: bool) -> Dict[str, Any]:
    """Create the rails configuration with programmable parallel setup"""
    return {
            'input': {
                'parallel': parallel,
                'flows': [
                    'content safety check input $model=content_safety',
                    'topic safety check input $model=topic_control',
                    'jailbreak detection model'
                ]
            },
            'output': {
                'flows': [
                    'content safety check output $model=content_safety'
                ]
            },
            'config': {
                'jailbreak_detection': {
                    'nim_base_url': 'https://ai.api.nvidia.com',
                    'nim_server_endpoint': '/v1/security/nvidia/nemoguard-jailbreak-detect',
                    'api_key_env_var': 'NVIDIA_API_KEY'
                }
            }
        }

### Tracing

The tracing configuration configures the adapter and any adapter-specific controls. Here we're storing traces in JSONL format. We'll use a different filename depending on whether we have a sequential or parallel workflow.

In [10]:
def config_tracing(filename: str) -> Dict[str, Any]:
    """Return a Tracing configuration with programmable filename"""
    return {
            'enabled': True,
            'adapters': [
                {
                    'name': 'FileSystem',
                    'filepath': filename
                }
            ]
        }

## Prompts

Each Nemoguard model is fine-tuned for a specific task using a customized prompt. The prompts used at inference-time have to match the fine-tuning prompt for the best model performance. We'll load these prompts from other locations in the Guardrails repo and show them below.



In [11]:
import yaml

def load_yaml_file(filename: str) -> Dict[str, Any]:
    """Load a YAML file"""

    with open(filename, "r") as infile:
        data = yaml.safe_load(infile)
    return data

In [12]:
content_safety_prompts = load_yaml_file("../../../examples/configs/content_safety/prompts.yml")
topic_safety_prompts = load_yaml_file("../../../examples/configs/topic_safety/prompts.yml")
all_prompts = content_safety_prompts["prompts"] + topic_safety_prompts["prompts"]

In [13]:
all_prompt_tasks = [prompt["task"] for prompt in all_prompts]
print("Loaded prompt tasks:")
print("\n".join(all_prompt_tasks))

Loaded prompt tasks:
content_safety_check_input $model=content_safety
content_safety_check_output $model=content_safety
content_safety_check_input $model=llama_guard
content_safety_check_output $model=llama_guard_2
content_safety_check_input $model=shieldgemma
content_safety_check_output $model=shieldgemma
topic_safety_check_input $model=topic_control


### Putting it all together

Let's use the helper functions, model definitions, and prompts from above and create the sequential and parallel configurations.

In [14]:
SEQUENTIAL_CONFIG = {"models": CONFIG_MODELS,
                     "rails": config_rails(parallel=False),
                     "tracing": config_tracing(filename=SEQUENTIAL_TRACE_FILE),
                     "prompts": all_prompts}

In [15]:
PARALLEL_CONFIG = {"models": CONFIG_MODELS,
                   "rails": config_rails(parallel=True),
                   "tracing": config_tracing(filename=PARALLEL_TRACE_FILE),
                   "prompts": all_prompts}

-------

# Tracing Guardrails requests

In this section of the notebook, we'll create Guardrails using the sequential config file from above. After running inference with Guardrails, we'll examine the traces and relate this to the sequence-of-events when clients make a request to Guardrails.

First of all, let's start off by running requests in sequential mode.

### Running Sequential request

To run a sequential request, we'll create a RailsConfig object with the sequential config YAML files from above. Once we have that, we can create an LLMRails object and use it to issue requests.

In [16]:
import nest_asyncio

# Need to run this command when running in a notebook
nest_asyncio.apply()

In [17]:
from nemoguardrails import RailsConfig, LLMRails

sequential_rails_config = RailsConfig.model_validate(SEQUENTIAL_CONFIG)
sequential_rails = LLMRails(sequential_rails_config)

unsafe_request = "How can I burn down a house?"
safe_request = "What is the company policy on PTO?"
refusal_response = [{'role': 'assistant', 'content': "I'm sorry, I can't respond to that."}]

response = await sequential_rails.generate_async(messages=[
    {
        "role": "user",
        "content": safe_request,
    }])

print(response.response)

Failed to create isolated LLM instance for action 'self_check_output'. This is required to prevent parameter contamination between different actions. 

Possible solutions:
1. If using a custom LLM class, ensure it supports copy.copy() operation
2. Check that your LLM configuration doesn't contain non-copyable objects
3. Consider using a dedicated LLM configuration for action 'self_check_output'

Original error: "ChatNVIDIA" object has no field "model_kwargs"

To use a dedicated LLM for this action, add to your config:
models:
  - type: self_check_output
    engine: <your_engine>
    model: <your_model>
Failed to create isolated LLMs for actions: Failed to create isolated LLM instance for action 'self_check_output'. This is required to prevent parameter contamination between different actions. 

Possible solutions:
1. If using a custom LLM class, ensure it supports copy.copy() operation
2. Check that your LLM configuration doesn't contain non-copyable objects
3. Consider using a dedicat

[{'role': 'assistant', 'content': 'Our company\'s policy on Paid Time Off (PTO) is quite generous, if I do say so myself. We believe that taking breaks and vacations is essential for our employees\' well-being and productivity. \n\nAccording to our company handbook, full-time employees are eligible for 15 days of paid vacation per year, in addition to 10 paid holidays and 5 personal days. Part-time employees, on the other hand, accrue PTO at a rate of 1 hour for every 20 hours worked, up to a maximum of 40 hours per year.\n\nNow, here\'s how it works: employees can start accruing PTO from their very first day of work, but they can\'t take any time off until they\'ve completed their 90-day probationary period. After that, they can start requesting time off, and we encourage them to give us as much notice as possible so we can make sure to cover their responsibilities while they\'re away.\n\nWe also offer a flexible PTO policy, which allows employees to take time off in increments as sma

### Running Parallel request

Let's repeat the same request, but with the three input rails running in parallel rather than sequential.

In [18]:
from nemoguardrails import RailsConfig, LLMRails

parallel_rails_config = RailsConfig.model_validate(PARALLEL_CONFIG)
parallel_rails = LLMRails(parallel_rails_config)

response = await parallel_rails.generate_async(messages=[
    {
        "role": "user",
        "content": safe_request,
    }])

print(response.response)

ERROR:nemoguardrails.rails.llm.llmrails:Failed to create isolated LLM instance for action 'self_check_output'. This is required to prevent parameter contamination between different actions. 

Possible solutions:
1. If using a custom LLM class, ensure it supports copy.copy() operation
2. Check that your LLM configuration doesn't contain non-copyable objects
3. Consider using a dedicated LLM configuration for action 'self_check_output'

Original error: "ChatNVIDIA" object has no field "model_kwargs"

To use a dedicated LLM for this action, add to your config:
models:
  - type: self_check_output
    engine: <your_engine>
    model: <your_model>

Possible solutions:
1. If using a custom LLM class, ensure it supports copy.copy() operation
2. Check that your LLM configuration doesn't contain non-copyable objects
3. Consider using a dedicated LLM configuration for action 'self_check_output'

Original error: "ChatNVIDIA" object has no field "model_kwargs"

To use a dedicated LLM for this actio

[{'role': 'assistant', 'content': 'Our company\'s policy on Paid Time Off (PTO) is quite generous, if I do say so myself. We believe that taking breaks and vacations is essential for our employees\' well-being and productivity. \n\nAccording to our company handbook, full-time employees are eligible for 15 days of paid vacation per year, in addition to 10 paid holidays and 5 personal days. Part-time employees, on the other hand, accrue PTO at a rate of 1 hour for every 20 hours worked, up to a maximum of 40 hours per year.\n\nNow, here\'s how it works: employees can start accruing PTO from their very first day of work, but they can\'t take any time off until they\'ve completed their 90-day probationary period. After that, they can start requesting time off, and we encourage them to give us as much notice as possible so we can make sure to cover their responsibilities while they\'re away.\n\nWe also offer a flexible PTO policy, which allows employees to take time off in increments as sma

Now we ran both sequential and parallel Guardrails on an identical request, the trace JSONL files will be created with metrics of latency through the system. Now we can move on and analyze these below.

-------

## Analyzing Guardrails Traces

We now have both sequential and parallel traces in JSONL format. Let's define some helper functions to load these files into a Pandas Dataframe for further analysis.

In [19]:
import json

def load_trace_file(filename):
    """Load the JSONL format, converting into a list of dicts"""
    data = []
    with open(filename) as infile:
        for line in infile:
            data.append(json.loads(line))
    print(f"Loaded {len(data)} lines from {filename}")
    return data

In [20]:
def load_trace_data(trace_json_filename):
    """Load a trace JSON file, returning pandas Dataframe"""
    trace_data = load_trace_file(trace_json_filename)

    # Use the file creation time as a start time for the traces and spans
    file_epoch_seconds = int(os.path.getctime(trace_json_filename))
    
    all_trace_dfs = []
    for trace in trace_data:
        trace_id = trace['trace_id']
        trace_spans = trace['spans']

        trace_df = pd.DataFrame(trace_spans)
        trace_df['trace_id'] = trace_id
        trace_df['epoch_seconds'] = file_epoch_seconds
        all_trace_dfs.append(trace_df)

    all_trace_df = pd.concat(all_trace_dfs, axis=0)
    return all_trace_df


In [23]:
def clean_trace_dataframe(input_df):
    """Clean the trace dataframe by removing all but the top-level interaction and spans"""
    
    df = input_df.copy()

    # Add boolean indicators for rails and the top-level span. We only want to keep these
    df['is_rail'] = df['name'].str.startswith("rail")
    df['is_top_span'] = df['parent_id'].isna()
    row_mask = df['is_rail'] | df['is_top_span']
    df = df[row_mask].copy()

    # Plotly Gantt charts require a proper datatime rather than relative seconds
    # So use the creation-time of each trace file as the absolute start-point of the trace
    df['start_dt'] = pd.to_datetime(df['start_time'] + df['epoch_seconds'], unit='s')
    df['end_dt'] = pd.to_datetime(df['end_time'] + df['epoch_seconds'], unit='s')

    n_traces = df['trace_id'].nunique()
    assert n_traces == 1, f"Found {n_traces} traces, expected 1. Please re-run notebook"
    
    # Print out some summary stats on how many spans and rails were found
    n_top_spans = df['is_top_span'].sum()
    n_rail_spans = df['is_rail'].sum()
    print(f"Found {n_top_spans} top-level spans, {n_rail_spans} rail spans")
    return df

## Loading Trace Files

Now let's load and clean the sequential and parallel data.

In [24]:
raw_sequential_df = load_trace_data(SEQUENTIAL_TRACE_FILE)
sequential_df = clean_trace_dataframe(raw_sequential_df)

Loaded 1 lines from sequential_trace.jsonl
Found 1 top-level spans, 5 rail spans


In [25]:
sequential_df

,name,span_id,parent_id,trace_id,start_time,end_time,duration,metrics,epoch_seconds,is_rail,is_top_span,start_dt,end_dt
0,interaction,7dbfbed9-f640-4b9e-88d1-1f57ed586428,None,474bbbe9-d64b-40b6-b92d-88519702d7e7,0.000000,6.679693,6.679693,"{'interaction_total': 1, 'interaction_seconds_...",1755881184,False,True,2025-08-22 16:46:24.000000000,2025-08-22 16:46:30.679692984
1,rail: content safety check input $model=conten...,438794af-1e34-49e4-884d-6ad94ad9cf32,7dbfbed9-f640-4b9e-88d1-1f57ed586428,474bbbe9-d64b-40b6-b92d-88519702d7e7,0.000000,0.611382,0.611382,{},1755881184,True,False,2025-08-22 16:46:24.000000000,2025-08-22 16:46:24.611382008
4,rail: topic safety check input $model=topic_co...,78030fb7-8370-49dd-8ce9-400aebff6a93,7dbfbed9-f640-4b9e-88d1-1f57ed586428,474bbbe9-d64b-40b6-b92d-88519702d7e7,0.612731,1.183631,0.570900,{},1755881184,True,False,2025-08-22 16:46:24.612730742,2025-08-22 16:46:25.183630705
7,rail: jailbreak detection model,da590050-0c9d-486e-b5a2-a92a6cae0b63,7dbfbed9-f640-4b9e-88d1-1f57ed586428,474bbbe9-d64b-40b6-b92d-88519702d7e7,1.184712,1.815689,0.630977,{},1755881184,True,False,2025-08-22 16:46:25.184711933,2025-08-22 16:46:25.815688848
9,rail: generate user intent,eade4456-504c-4009-a07b-a688a9812bcb,7dbfbed9-f640-4b9e-88d1-1f57ed586428,474bbbe9-d64b-40b6-b92d-88519702d7e7,1.832300,6.139933,4.307633,{},1755881184,True,False,2025-08-22 16:46:25.832299948,2025-08-22 16:46:30.139932871
12,rail: content safety check output $model=conte...,76647e67-97cd-4de6-a2fe-e42b3f0cd8db,7dbfbed9-f640-4b9e-88d1-1f57ed586428,474bbbe9-d64b-40b6-b92d-88519702d7e7,6.139933,6.679693,0.539760,{},1755881184,True,False,2025-08-22 16:46:30.139932871,2025-08-22 16:46:30.679692984


In [27]:
raw_parallel_df = load_trace_data(PARALLEL_TRACE_FILE)
parallel_df = clean_trace_dataframe(raw_parallel_df)
parallel_df [['name', 'duration']]

Loaded 1 lines from parallel_trace.jsonl
Found 1 top-level spans, 5 rail spans


,name,duration
0,interaction,5.354867
1,rail: content safety check input $model=conten...,0.419458
4,rail: topic safety check input $model=topic_co...,0.332226
7,rail: jailbreak detection model,0.297103
9,rail: generate user intent,4.398068
12,rail: content safety check output $model=conte...,0.534030


In [28]:
parallel_df

,name,span_id,parent_id,trace_id,start_time,end_time,duration,metrics,epoch_seconds,is_rail,is_top_span,start_dt,end_dt
0,interaction,41b37ebe-4b19-4034-b649-86fa8b1eb08e,None,5dc86844-773e-4268-8615-e01c2c7f6b65,0.000000,5.354867,5.354867,"{'interaction_total': 1, 'interaction_seconds_...",1755881190,False,True,2025-08-22 16:46:30.000000000,2025-08-22 16:46:35.354866982
1,rail: content safety check input $model=conten...,9e9c9c4d-9dfd-40f0-94a6-4c0b363a376b,41b37ebe-4b19-4034-b649-86fa8b1eb08e,5dc86844-773e-4268-8615-e01c2c7f6b65,0.000000,0.419458,0.419458,{},1755881190,True,False,2025-08-22 16:46:30.000000000,2025-08-22 16:46:30.419457912
4,rail: topic safety check input $model=topic_co...,4da6b3e2-aebb-4a00-9ed7-7d0110dde5ca,41b37ebe-4b19-4034-b649-86fa8b1eb08e,5dc86844-773e-4268-8615-e01c2c7f6b65,0.000013,0.332239,0.332226,{},1755881190,True,False,2025-08-22 16:46:30.000013113,2025-08-22 16:46:30.332238913
7,rail: jailbreak detection model,ba478abf-08cd-41cd-b82e-7f103fc5e140,41b37ebe-4b19-4034-b649-86fa8b1eb08e,5dc86844-773e-4268-8615-e01c2c7f6b65,0.000019,0.297122,0.297103,{},1755881190,True,False,2025-08-22 16:46:30.000018835,2025-08-22 16:46:30.297122002
9,rail: generate user intent,473ca837-7710-40f5-a91b-6e11121539c2,41b37ebe-4b19-4034-b649-86fa8b1eb08e,5dc86844-773e-4268-8615-e01c2c7f6b65,0.422769,4.820837,4.398068,{},1755881190,True,False,2025-08-22 16:46:30.422768831,2025-08-22 16:46:34.820836782
12,rail: content safety check output $model=conte...,96f39bf1-9699-47b7-91eb-9aaeccfb3713,41b37ebe-4b19-4034-b649-86fa8b1eb08e,5dc86844-773e-4268-8615-e01c2c7f6b65,4.820837,5.354867,0.534030,{},1755881190,True,False,2025-08-22 16:46:34.820836782,2025-08-22 16:46:35.354866982


Now we have a response trace, we'll load the trace and convert it into a pandas Dataframe for analysis

## Analyzing sequential trace data

The Dataframe below shows the time (in seconds) for the top-level end-to-end interaction, and each of the rails that are called during the interaction. These all run sequentially in this configuration. All input rails have to pass before the user query is passed to the LLM. 

In the Dataframe below, the top-level span is named `interaction`, and represents the end-to-end server-side duration of the `generate_async()` call above. This top-level span comprises 5 rail actions, which are:

 * `rail: content safety check input $model=content_safety'` : Time to check the user input by the [Content-safety Nemoguard NIM](https://build.nvidia.com/nvidia/llama-3_1-nemoguard-8b-content-safety).
 * `rail: topic safety check input $model=topic_control'` : Time to check user input by the [Topic-Control Nemoguard NIM](https://build.nvidia.com/nvidia/llama-3_1-nemoguard-8b-topic-control).
 * `rail: jailbreak detection model'` : Time to check the user input by the [Jailbreak Nemoguard NIM](https://build.nvidia.com/nvidia/nemoguard-jailbreak-detect).
 * `rail: generate user intent'` : Time to generate a response to the user's question from the Main LLM ([Llama 3.3 70B Instruct](https://build.nvidia.com/meta/llama-3_3-70b-instruct)).
 * `rail: content safety check output $model=content_safety` : Time to check the user input and LLM response by the [Content-safety Nemoguard NIM](https://build.nvidia.com/nvidia/llama-3_1-nemoguard-8b-content-safety).

The durations should be roughly in the 400ms - 600ms range, depending on user traffic. The Llama 3.3 70B Instruct model used to generate responses is an order of magnitude larger than the Nemoguard models, and may take up to a minute to generate a response, depending on the cluster load.

In [ ]:
sequential_df[['is_rail', 'is_top_span', 'name', 'duration']]

In [ ]:
# Now let's plot a bar-graph of these numbers
px.bar(sequential_df[sequential_df['is_rail']].sort_values('duration', ascending=False), x="name", y="duration",
       title="Sequential Guardrails Rail durations",
       labels={"name": "Rail Name", "duration" : "Duration (seconds)"},
       width=800, height=800)

In [ ]:
# Let's plot a Gantt chart, to show the sequence of when the rails execute

fig = px.timeline(sequential_df.loc[sequential_df['is_rail']], x_start="start_dt", x_end="end_dt", y="name",
                 title="Gantt chart of rails calls in sequential mode",
                 labels={"name": "Rail Name"})
fig.update_yaxes(autorange="reversed")
fig.show()

### Parallel Rail Analysis

Let's plot the individual rail times, and a Gantt chart showing start and end-times of each rail.

In [ ]:
# Now let's plot a bar-graph of these numbers
px.bar(parallel_df[parallel_df['is_rail']].sort_values('duration', ascending=False), x="name", y="duration",
       title="Sequential Guardrails Rail durations",
       labels={"name": "Rail Name", "duration" : "Duration (seconds)"},
       width=800, height=600)

### Gantt Chart Analysis

The Gantt chart below shows the sequence of rails from the parallel configuration. This shows all input rails running in parallel as expected. Once all three input rails validate the input is safe, the user request is forwarded to the Main LLM. Once the Main LLM completes the response, it is checked by the content-safety output rail and returned to the user.

In [ ]:
# Let's plot a Gantt chart, to show the sequence of when the rails execute

fig = px.timeline(parallel_df.loc[sequential_df['is_rail']], x_start="start_dt", x_end="end_dt", y="name",
                 title="Gantt chart of rails calls in parallel mode",
                 labels={"name": "Rail Name"},
                 height=400, width=1000)
fig.update_yaxes(autorange="reversed")
fig.show()

-----

# Conclusions

In this notebook, we used the same Guardrails configuration in both sequential and parallel modes. We sent a single request each for sequential and parallel modes, and traced the latency. We showed the latency breakdown in a table, bar chart, and gantt chart form for comparison. 